# 2014 FIFA World Cup | Macroeconomic and Tourism Impact Analysis
*Case Study: Brazil (Host Country) vs. Mexico (Control Group)*

This notebook tests whether mega‑sporting events create sustained economic benefits. It assesses if Brazil’s 2014 FIFA World Cup hosting led to long‑term growth or only a temporary spike, using México as the control case.

### Project Overview & Methodology
- Data Collection: Automatically retrieves macroeconomic and tourism indicators from the World Bank API, along with government gross debt data from the IMF API.
- Web Scraping: Uses BeautifulSoup to extract official visual assets (such as event logos) directly from Wikipedia.
- Comparative Analysis: Merges, aggregates, and computes percentage‑based statistical changes across two key periods: Pre‑World Cup (2009–2013) and Post‑World Cup (2015–2019).

### Key Indicators
The notebook compares Brazil (BRA) and Mexico (MEX) across several core metrics:
- Economic Health: Real GDP, GDP growth rate, and inflation (CPI).
- Labor Market: Unemployment rates.
- Fiscal Position: Government gross debt as a percentage of GDP.
- Tourism Performance: International tourist arrivals and tourism receipts.

We analyze different World Cups to assess their macroeconomic impact on the host country. A control country with similar structural characteristics is used for comparison.

In [38]:
# IMPORT LIBRARIES AND DISPLAY SETTINGS


# pandas and numpy support data manipulation and calculations
import pandas as pd
import numpy as np
# requests retrieves data from the World Bank and IMF APIs
import requests
# matplotlib supports later visual analysis - only ussed for certain solutions
import matplotlib.pyplot as plt

# Display large numbers with commas and two decimal places
pd.set_option("display.float_format", "{:,.2f}".format)


In [39]:
# WORLD BANK API HELPER FUNCTION
# Retrieve one indicator from the World Bank API and prepare it for analysis

def get_worldbank_indicator(country, indicator, metric_name):

    # Build the API request using the country and indicator codes
    url_worldbank = f"https://api.worldbank.org/v2/country/{country}/indicator/{indicator}?format=json&per_page=100"

    # Request the data and stop if the API returns an error
    response = requests.get(url_worldbank)
    response.raise_for_status()

    # Convert the API response into a two-column DataFrame
    data = response.json()

    df_country = pd.DataFrame(data[1])

    df_country = df_country[["date", "value"]]

    df_country = df_country.rename(columns={
        "date": "year",
        "value": metric_name
    })

    # Convert year to an integer and keep the 2009-2019 study period
    df_country["year"] = df_country["year"].astype(int)

    df_country = df_country[df_country["year"].between(2009, 2019)]

    return df_country.sort_values("year").reset_index(drop=True)

In [40]:
# IMF API HELPER FUNCTION
# Retrieve one indicator from the IMF DataMapper API

def get_IMF_indicator(country, indicator, metric_name):

    # Build and send the request for the selected IMF indicator
    url_IMF = f"https://www.imf.org/external/datamapper/api/v2/{indicator}"

    response = requests.get(url_IMF)
    response.raise_for_status()

    data = response.json()

    # Select the year-value dictionary for the requested country
    country_data = data["values"][indicator][country]

    df_IMF_country = pd.DataFrame(
        country_data.items(),
        columns=["year", metric_name]
    )

    # Convert year to an integer and keep the 2009-2019 study period
    df_IMF_country["year"] = df_IMF_country["year"].astype(int)

    df_IMF_country = df_IMF_country[
        df_IMF_country["year"].between(2009, 2019)
    ]

    return df_IMF_country.sort_values("year").reset_index(drop=True)

In [41]:
# BUILD EACH COUNTRY'S DATASET
# Build one complete macroeconomic and tourism dataset for a country

def build_country_dataset(wb_code, imf_code, country_name):

    # Actual GDP measured in current US dollars
    gdp = get_worldbank_indicator(
        wb_code,
        "NY.GDP.MKTP.CD",
        "gdp_current_usd"
    )

    # Annual GDP growth rate expressed as a percentage
    gdp_growth = get_worldbank_indicator(
        wb_code,
        "NY.GDP.MKTP.KD.ZG",
        "gdp_growth_pct"
    )

    # Annual inflation rate expressed as a percentage
    inflation = get_worldbank_indicator(
        wb_code,
        "FP.CPI.TOTL.ZG",
        "inflation_pct"
    )

    # Unemployment as a percentage of the labour force
    unemployment = get_worldbank_indicator(
        wb_code,
        "SL.UEM.TOTL.ZS",
        "unemployment_pct"
    )

    #Tourist Arrivals
    tourist_arrivals = get_worldbank_indicator(
        wb_code,
        "ST.INT.ARVL",
        "tourist_arrivals"
    )

    # International tourism revenue measured in current US dollars
    tourism_receipts = get_worldbank_indicator(
        wb_code,
        "ST.INT.RCPT.CD",
        "tourism_revenue_current_usd"
    )

    # Government debt expressed as a percentage of GDP
    government_debt = get_IMF_indicator(
        imf_code,
        "GGXWDG_NGDP",
        "government_debt_pct_gdp"
    )

    # Merge all indicators by year; outer joins preserve incomplete years
    country_df = (
        gdp
        .merge(gdp_growth, on="year", how="outer")
        .merge(inflation, on="year", how="outer")
        .merge(unemployment, on="year", how="outer")
        .merge(tourist_arrivals, on="year", how="outer")
        .merge(tourism_receipts, on="year", how="outer")
        .merge(government_debt, on="year", how="outer")
    )

    # Calculate how much tourism revenue contributes relative to GDP
    country_df["tourism_revenue_pct_gdp"] = (
        country_df["tourism_revenue_current_usd"]
        / country_df["gdp_current_usd"]
        * 100
    )

    # Place the country name immediately after the year column
    country_df.insert(1, "country", country_name)

    return country_df

In [42]:
# USING BRAZIL AS HOST AND MEXICO AS CONTROL AND COMBINING BOTH DATAS
# Build the individual country datasets using their API country codes

brazil_df = build_country_dataset("BRA", "BRA", "Brazil")
mexico_df = build_country_dataset("MEX", "MEX", "Mexico")

# Stack both countries into one analysis-ready table
brazil_mexico_df = pd.concat(
    [brazil_df, mexico_df],
    ignore_index=True
)

# Organize the rows by country and year
brazil_mexico_df = brazil_mexico_df.sort_values(
    ["country", "year"]
).reset_index(drop=True)

brazil_mexico_df

,year,country,gdp_current_usd,gdp_growth_pct,inflation_pct,unemployment_pct,tourist_arrivals,tourism_revenue_current_usd,government_debt_pct_gdp,tourism_revenue_pct_gdp
0,2009,Brazil,"1,666,996,294,252.12",-0.13,4.89,9.42,"4,802,000.00","5,635,000,000.00",64.70,0.34
1,2010,Brazil,"2,208,838,108,484.35",7.53,5.04,8.42,"5,161,000.00","5,522,000,000.00",62.40,0.25
2,2011,Brazil,"2,616,156,606,579.21",3.97,6.64,7.58,"5,433,000.00","6,370,000,000.00",60.60,0.24
3,2012,Brazil,"2,465,228,293,706.86",1.92,5.40,7.25,"5,677,000.00","6,623,000,000.00",61.60,0.27
4,2013,Brazil,"2,472,819,362,043.74",3.00,6.20,7.07,"5,813,000.00","6,784,000,000.00",59.60,0.27
5,2014,Brazil,"2,456,043,766,032.38",0.50,6.33,6.75,"6,430,000.00","7,405,000,000.00",61.60,0.30
6,2015,Brazil,"1,802,211,999,456.42",-3.55,9.03,8.54,"6,306,000.00","6,254,000,000.00",71.70,0.35
7,2016,Brazil,"1,795,693,265,999.04",-3.28,8.74,11.58,"6,547,000.00","6,613,000,000.00",77.40,0.37
8,2017,Brazil,"2,063,514,688,805.78",1.32,3.45,12.79,"6,589,000.00","6,175,000,000.00",82.70,0.30
9,2018,Brazil,"1,916,933,708,352.71",1.78,3.66,12.33,"6,621,000.00","6,324,000,000.00",84.80,0.33


In [45]:
# CREATE BEFORE/AFTER PERIODS FROM THE YEAR COLUMN

# Years before the 2014 World Cup are labeled "Before"
# Years from 2014 onward are labeled "After"

brazil_mexico_df2 = brazil_mexico_df.copy()

brazil_mexico_df2["period"] = np.where(
    brazil_mexico_df2["year"] < 2014,
    "Before",
    "After"
)

brazil_mexico_df2

,year,country,gdp_current_usd,gdp_growth_pct,inflation_pct,unemployment_pct,tourist_arrivals,tourism_revenue_current_usd,government_debt_pct_gdp,tourism_revenue_pct_gdp,period
0,2009,Brazil,"1,666,996,294,252.12",-0.13,4.89,9.42,"4,802,000.00","5,635,000,000.00",64.70,0.34,Before
1,2010,Brazil,"2,208,838,108,484.35",7.53,5.04,8.42,"5,161,000.00","5,522,000,000.00",62.40,0.25,Before
2,2011,Brazil,"2,616,156,606,579.21",3.97,6.64,7.58,"5,433,000.00","6,370,000,000.00",60.60,0.24,Before
3,2012,Brazil,"2,465,228,293,706.86",1.92,5.40,7.25,"5,677,000.00","6,623,000,000.00",61.60,0.27,Before
4,2013,Brazil,"2,472,819,362,043.74",3.00,6.20,7.07,"5,813,000.00","6,784,000,000.00",59.60,0.27,Before
5,2014,Brazil,"2,456,043,766,032.38",0.50,6.33,6.75,"6,430,000.00","7,405,000,000.00",61.60,0.30,After
6,2015,Brazil,"1,802,211,999,456.42",-3.55,9.03,8.54,"6,306,000.00","6,254,000,000.00",71.70,0.35,After
7,2016,Brazil,"1,795,693,265,999.04",-3.28,8.74,11.58,"6,547,000.00","6,613,000,000.00",77.40,0.37,After
8,2017,Brazil,"2,063,514,688,805.78",1.32,3.45,12.79,"6,589,000.00","6,175,000,000.00",82.70,0.30,After
9,2018,Brazil,"1,916,933,708,352.71",1.78,3.66,12.33,"6,621,000.00","6,324,000,000.00",84.80,0.33,After


In [50]:
# CALCULATE AVERAGE INDICATORS BY COUNTRY AND PERIOD
# Select the indicators used to compare the Before and After periods

metrics = [
    "gdp_growth_pct",
    "inflation_pct",
    "unemployment_pct",
    "government_debt_pct_gdp",
    "tourism_revenue_current_usd",
    "tourist_arrivals"
]

# Calculate the mean of each indicator for every country-period combination
avg_table = brazil_mexico_df2.pivot_table(
    index="country",
    columns="period",
    values=metrics,
    aggfunc="mean"
)

# Round the displayed results while keeping the underlying values unchanged
avg_table.round(2)

gdp_growth_pct        government_debt_pct_gdp        inflation_pct  \
period           After Before                   After Before         After   
country                                                                      
Brazil           -0.33   3.26                   77.55  61.78          5.82   
Mexico            1.74   1.31                   51.62  41.60          4.02   

               tourism_revenue_current_usd                   tourist_arrivals  \
period  Before                       After            Before            After   
country                                                                         
Brazil    5.63            6,483,000,000.00  6,186,800,000.00     6,474,333.33   
Mexico    4.16           21,345,000,000.00 13,051,800,000.00    92,712,666.67   

                      unemployment_pct         
period         Before            After Before  
country                                        
Brazil   5,377,200.00            10.66   7.95  
Mexico  80,115,600.00             3.86   5.13

In [51]:
# MEASURE THE CHANGE FROM BEFORE TO AFTER
# Extract the country averages for each period from the pivot table

#Cross sections (xs) works this way:
    #   dataframe.xs(
    #       label_to_find,
    #       level=where_to_find_it, --> in this case
    #        axis=rows_or_columns
    #   )

before = avg_table.xs(
    "Before",
    level=1,
    axis=1
)

after = avg_table.xs(
    "After",
    level=1,
    axis=1
)

# Positive values indicate an increase; negative values indicate a decrease
change_table = after - before

change_table.round(2)

,gdp_growth_pct,government_debt_pct_gdp,inflation_pct,tourism_revenue_current_usd,tourist_arrivals,unemployment_pct
country,,,,,,
Brazil,-3.59,15.77,0.19,"296,200,000.00","1,097,133.33",2.71
Mexico,0.43,10.02,-0.13,"8,293,200,000.00","12,597,066.67",-1.27


In [52]:
# ESTIMATE BRAZIL'S RELATIVE WORLD CUP IMPACT
# Subtract Mexicos's change from Brazil's change to create a comparison group

impact_table = (
    change_table.loc["Brazil"]
    -
    change_table.loc["Mexico"]
)

# Convert the resulting Series into a clearly labelled one-column table
impact_table = impact_table.to_frame(
    name="Brazil_minus_Mexico"
)

impact_table.round(2)

,Brazil_minus_Mexico
gdp_growth_pct,-4.03
government_debt_pct_gdp,5.75
inflation_pct,0.32
tourism_revenue_current_usd,"-7,997,000,000.00"
tourist_arrivals,"-11,499,933.33"
unemployment_pct,3.98


In [54]:
# Create a GDP table with years as rows and countries as columns
# This makes it easy to compare Germany and France GDP year by year

gdp_table = brazil_mexico_df2.pivot_table(
    index="year",
    columns="country",
    values="gdp_current_usd",
    aggfunc="mean"
)

gdp_table

country,Brazil,Mexico
year,,
2009,"1,666,996,294,252.12","943,437,415,024.63"
2010,"2,208,838,108,484.35","1,105,424,238,731.09"
2011,"2,616,156,606,579.21","1,229,013,703,416.76"
2012,"2,465,228,293,706.86","1,255,110,424,817.79"
2013,"2,472,819,362,043.74","1,327,436,290,282.67"
2014,"2,456,043,766,032.38","1,364,507,717,614.13"
2015,"1,802,211,999,456.42","1,213,294,467,716.88"
2016,"1,795,693,265,999.04","1,112,233,497,452.70"
2017,"2,063,514,688,805.78","1,190,721,475,906.00"


In [55]:
# Create a tourism revenue table with years as rows and countries as columns
# This shows how much tourism revenue each country earned each year

tourism_revenue_table = brazil_mexico_df2.pivot_table(
    index="year",
    columns="country",
    values="tourism_revenue_current_usd",
    aggfunc="mean"
)

tourism_revenue_table

country,Brazil,Mexico
year,,
2009,"5,635,000,000.00","12,542,000,000.00"
2010,"5,522,000,000.00","12,628,000,000.00"
2011,"6,370,000,000.00","12,458,000,000.00"
2012,"6,623,000,000.00","13,320,000,000.00"
2013,"6,784,000,000.00","14,311,000,000.00"
2014,"7,405,000,000.00","16,606,000,000.00"
2015,"6,254,000,000.00","18,729,000,000.00"
2016,"6,613,000,000.00","20,619,000,000.00"
2017,"6,175,000,000.00","22,467,000,000.00"


In [56]:
# Create a tourist arrivals table with years as rows and countries as columns
# This compares the number of international tourist arrivals each year

tourist_arrivals_table = brazil_mexico_df.pivot_table(
    index="year",
    columns="country",
    values="tourist_arrivals",
    aggfunc="mean"
)

tourist_arrivals_table

country,Brazil,Mexico
year,,
2009,"4,802,000.00","88,044,000.00"
2010,"5,161,000.00","81,953,000.00"
2011,"5,433,000.00","75,732,000.00"
2012,"5,677,000.00","76,749,000.00"
2013,"5,813,000.00","78,100,000.00"
2014,"6,430,000.00","81,042,000.00"
2015,"6,306,000.00","87,129,000.00"
2016,"6,547,000.00","94,853,000.00"
2017,"6,589,000.00","99,349,000.00"


In [57]:
# Create a GDP growth table with years as rows and countries as columns
# This compares each country's yearly GDP growth percentage

gdp_growth_table = brazil_mexico_df2.pivot_table(
    index="year",
    columns="country",
    values="gdp_growth_pct",
    aggfunc="mean"
)

gdp_growth_table

country,Brazil,Mexico
year,,
2009,-0.13,-6.30
2010,7.53,4.97
2011,3.97,3.44
2012,1.92,3.55
2013,3.00,0.85
2014,0.50,2.50
2015,-3.55,2.70
2016,-3.28,1.77
2017,1.32,1.87


In [59]:
# Create a government debt table with years as rows and countries as columns
# This compares government debt as a percentage of GDP for each country

gvt_debt= brazil_mexico_df2.pivot_table(
    index="year",
    columns= ["country"],
    values="government_debt_pct_gdp",
)

gvt_debt

country,Brazil,Mexico
year,,
2009,64.70,41.70
2010,62.40,40.20
2011,60.60,41.20
2012,61.60,40.80
2013,59.60,44.10
2014,61.60,47.10
2015,71.70,51.00
2016,77.40,55.00
2017,82.70,52.50


In [60]:
# Create a tourism revenue percentage-of-GDP table by year and country
# This shows how important tourism revenue was relative to each country's GDP

tourism_pct_gdp = brazil_mexico_df.pivot_table(
    index="year",
    columns= ["country"],
    values="tourism_revenue_pct_gdp",
)

tourism_pct_gdp

country,Brazil,Mexico
year,,
2009,0.34,1.33
2010,0.25,1.14
2011,0.24,1.01
2012,0.27,1.06
2013,0.27,1.08
2014,0.30,1.22
2015,0.35,1.54
2016,0.37,1.85
2017,0.30,1.89


In [61]:
# Function to find the percentage change between 2 years after World Cup in a dataframe.

def calculate_percent_change(df, start_year, end_year, value_column):
    start_value = df.loc[df["year"] == start_year, value_column].iloc[0]
    end_value = df.loc[df["year"] == end_year, value_column].iloc[0]

    return ((end_value - start_value) / start_value) * 100

In [62]:
calculate_percent_change(brazil_df,2015, 2019,"gdp_current_usd" )

np.float64(3.9438289947934995)

In [64]:
# WEB SCRAPING FOR 2014 WORLD CUP OFFICIAL LOGO

from bs4 import BeautifulSoup

# Step 1 URL
url = "https://en.wikipedia.org/wiki/2014_FIFA_World_Cup"

# Step 2 User Agent Header
headers = {"User-Agent" : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari 537.36"}

# Step 3a Make the Requests and save the Response
wiki_response = requests.get(url, headers=headers) 

In [65]:
#  Step 3b Test status code (Good practice)

if wiki_response.status_code == 200:
    print("Ok")
else:
    print("Not ok")

Ok


In [66]:
# Step 4 Parse the Response
wiki_soup = BeautifulSoup(wiki_response.content, "html.parser")

# Data Step 1 - Look at the HTML and search for all images. We're looking for the logo.
all_images = wiki_soup.find_all("img")
print(len(all_images))

686


In [67]:
# Verify the contents
from IPython.display import Image, display

for img in all_images:
    src = img.get("src")

    if src and "2014_FIFA_World_Cup.svg" in src:
        img_url = "https:" + src
        display(Image(url=img_url))